# Task 3 — RAG Pipeline: Retrieval, Generation & Evaluation

**Goal:** load the vector store built in notebook 02, wire up the retriever and the
Groq LLM generator into a full RAG pipeline, run it against a set of representative
questions, and produce the qualitative evaluation table for the report.

Logic lives in `src/retriever.py`, `src/generator.py`, `src/prompt_templates.py`, and
`src/rag_pipeline.py` — this notebook demonstrates and evaluates it, it doesn't
reimplement it.

**Requires:** `vector_store/chroma_db/` populated (notebook 02) and `GROQ_API_KEY`
set in `.env`.


In [5]:
import sys
sys.path.append("..")

import pandas as pd
from IPython.display import display, Markdown

from src import config
from src.retriever import Retriever
from src.generator import generate
from src.prompt_templates import build_prompt
from src.rag_pipeline import RAGPipeline
from src.embedding import get_chroma_collection


pd.set_option("display.max_colwidth", 200)

## 1. Vector store sanity check

Confirm the ChromaDB collection built in notebook 02 is actually populated before
we build anything on top of it.


In [6]:
collection = get_chroma_collection()
print(f"Collection: {config.CHROMA_COLLECTION_NAME}")
print(f"Chunks indexed: {collection.count():,}")

assert collection.count() > 0, "Vector store is empty — run notebook 02 first."

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection: complaint_chunks
Chunks indexed: 35,517


## 2. Retriever in isolation

Before wiring in the LLM, confirm retrieval alone is returning relevant, on-topic
chunks. If retrieval is weak, no amount of prompt engineering fixes the final answer
— garbage in, garbage out.


In [7]:
retriever = Retriever(top_k=5)

test_question = "Why are people unhappy with Credit Cards?"
chunks = retriever.retrieve(test_question)

for i, c in enumerate(chunks):
    print(f"[{i+1}] similarity={1 - c['distance']:.3f}  product={c['metadata']['product_category']}  issue={c['metadata']['issue']}")
    print(c["text"][:200])
    print()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[1] similarity=0.646  product=Credit card  issue=Other features, terms, or problems
. nice thin and non plastic greasy credit. they are obviously and don't like customers. why are they sending new credit cards like this?

[2] similarity=0.623  product=Credit card  issue=Fees or interest
. further, they try to promote use of their credit cards, for precisely this reason is my guess. please help!

[3] similarity=0.619  product=Credit card  issue=Closing your account
. most of the cards started out with little available credit and as i got established with them, they kept increasing my credit limits as well as giving me more credit cards. i have been an exceptiona

[4] similarity=0.607  product=Credit card  issue=Getting a credit card
. i again emphasized that i was not interested in a fancy, benefits laden credit card. rather, i wanted something simple and low risk, so that the underwriters would be more likely to approve it. i wa

[5] similarity=0.603  product=Credit card  issue=Problem

### Retrieval with a product filter

The retriever also supports filtering to a single product category — useful when a
PM already knows which product they're investigating and wants to exclude noise
from the others.


In [8]:
filtered_chunks = retriever.retrieve("unauthorized transactions", product_filter="Money transfer")

for c in filtered_chunks[:3]:
    print(f"[{c['metadata']['product_category']}] similarity={1 - c['distance']:.3f}")
    print(c["text"][:180])
    print()

[Money transfer] similarity=0.731
unauthorized transaction on account and i did not authorize

[Money transfer] similarity=0.706
i have been getting unauthorized transactions taking out my account with no refund for internet purchases and ect..

[Money transfer] similarity=0.640
. this person by the name of was able to activate my new card. he then proceeded to tell me that it appears someone is trying to illegally wire money from my account. so i proceede



## 3. Prompt template

The exact prompt sent to the LLM — grounded-only, admits uncertainty, cites how many
excerpts support a claim. See `src/prompt_templates.py`.


In [9]:
example_prompt = build_prompt(test_question, [c["text"] for c in chunks[:2]])
print(example_prompt)

You are a financial analyst assistant for CrediTrust, a digital finance company. Internal teams (Product, Support, Compliance) rely on you to summarize patterns in customer complaints accurately.

Rules:
1. Base your answer ONLY on the retrieved complaint excerpts provided below. Do not use outside knowledge.
2. If the excerpts don't contain enough information to answer, say so explicitly - do not guess.
3. When you identify a trend, mention roughly how many of the retrieved excerpts support it.
4. Write concisely, in plain English, for a non-technical Product Manager.
5. Do not fabricate complaint details, dates, or company names that aren't in the context.

Context (retrieved complaint excerpts):
[Excerpt 1]
. nice thin and non plastic greasy credit. they are obviously and don't like customers. why are they sending new credit cards like this?

[Excerpt 2]
. further, they try to promote use of their credit cards, for precisely this reason is my guess. please help!

Question: Why are p

## 4. Generation — single question, full trace

Run one question through retrieval + generation manually so you can see each stage
before running the full pipeline in bulk.


In [10]:
answer = generate(test_question, [c["text"] for c in chunks])

display(Markdown(f"**Q:** {test_question}\n\n**A:** {answer}"))

**Q:** Why are people unhappy with Credit Cards?

**A:** Based on the retrieved complaint excerpts, people are unhappy with credit cards for several reasons. 

About 2 excerpts suggest that customers are unhappy with the type of credit cards being offered or promoted, with one customer feeling pressured into taking a card with more benefits than they wanted, and another expressing frustration with the cards being sent to them.

Around 2 excerpts indicate that customers are dissatisfied with how their credit limits and accounts are being managed, with one customer feeling that their credit limits were increased too much and another feeling that their credit score was not being taken into account.

1 excerpt suggests that a customer is unhappy with the way their lost or stolen card was handled, and the subsequent harassment they received from the company.

It's worth noting that these excerpts may not be representative of all customer complaints, and more information would be needed to fully understand the scope of the issues.

## 5. Full pipeline — `RAGPipeline.ask()`

This is the exact call the FastAPI backend makes. One method, retrieval + generation
+ sources, ready to hand to a UI.


In [11]:
pipeline = RAGPipeline()

result = pipeline.ask("What are the most common Personal Loan complaints?")

display(Markdown(f"**Answer:**\n\n{result['answer']}"))
print(f"\nGrounded in {len(result['sources'])} source excerpts:")
for s in result["sources"][:2]:
    print(f"  - #{s['metadata']['complaint_id']}: {s['text'][:120]}...")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


**Answer:**

Based on the retrieved complaint excerpts, the most common Personal Loan complaints are:

1. High interest rates (supported by 3 excerpts: Excerpt 1 mentions a 25% interest rate, Excerpt 2 calls the interest rates "appalling", and Excerpt 4 mentions doubled interest).
2. Poor customer service, with representatives being described as "rude" and "unprofessional" (supported by 3 excerpts: Excerpt 2, Excerpt 3, and Excerpt 4).
3. Aggressive collection practices, including harassment and threats (supported by 1 excerpt: Excerpt 4, but it's a significant concern).

Note that Excerpt 5 is more of an inquiry about potential issues with a loan, rather than a direct complaint. Overall, about 4 out of 5 excerpts mention some kind of issue with the loan or the company's behavior.


Grounded in 5 source excerpts:
  - #11756890: is the complainant against one main financial, , ny , ph . i borrowed 10000.00 personal loan, i knew the interest rate w...
  - #3871911: . they are rude and unprofessional and their interest rates are appalling. i paid at least a month for almost 12 months ...


## 6. Qualitative evaluation — the Task 3 deliverable

Run the pipeline against 10 representative questions and score each answer 1-5 on
groundedness and usefulness to a PM. This cell builds the table; fill in the
`Quality Score` and `Comments` columns after reading each answer.


In [12]:
eval_questions = [
    "Why are people unhappy with Credit Cards?",
    "What are the most common complaints about Personal Loans?",
    "Are customers reporting unauthorized transactions in Money Transfers?",
    "What issues come up most with Savings Accounts?",
    "Do complaints mention long resolution times?",
    "Are there complaints about hidden fees across any product?",
    "What fraud-related patterns appear in the complaints?",
    "How do customers describe problems with customer service?",
    "Are there recurring complaints about a specific company?",
    "What would you flag as the top 3 emerging issues this month?",
]

eval_rows = []
for q in eval_questions:
    result = pipeline.ask(q)
    top_sources = "; ".join(
        f"#{s['metadata']['complaint_id']}: {s['text'][:100]}..." for s in result["sources"][:2]
    )
    eval_rows.append({
        "Question": q,
        "Generated Answer": result["answer"],
        "Retrieved Sources (top 2)": top_sources,
        "Quality Score (1-5)": "",   # fill in manually after reading
        "Comments / Analysis": "",   # fill in manually after reading
    })

eval_df = pd.DataFrame(eval_rows)
eval_df

,Question,Generated Answer,Retrieved Sources (top 2),Quality Score (1-5),Comments / Analysis
0,Why are people unhappy with Credit Cards?,"Based on the retrieved complaint excerpts, people are unhappy with credit cards for several reasons. \n\nAbout 2 excerpts suggest that customers are unhappy with the type of credit cards being off...","#3271275: . nice thin and non plastic greasy credit. they are obviously and don't like customers. why are they...; #2888328: . further, they try to promote use of their credit cards, for precisely...",,
1,What are the most common complaints about Personal Loans?,"Based on the retrieved complaint excerpts, the most common complaints about Personal Loans are:\n\n* High interest rates (supported by Excerpts 1 and 4, roughly 2 out of 5 excerpts)\n* Unapproved ...",#3871911: . they are rude and unprofessional and their interest rates are appalling. i paid at least a month f...; #8625161: i heard that one main financial added services and did other things on ...,,
2,Are customers reporting unauthorized transactions in Money Transfers?,"Yes, customers are reporting unauthorized transactions in Money Transfers. About 4 out of the 5 retrieved excerpts mention unauthorized transactions, with specific details such as stolen access to...","#11260307: . they could not give me that information, but it apparently is a service the bank provides. now may...; #8883886: i reported unauthorized transactions that occurred on my debit card wh...",,
3,What issues come up most with Savings Accounts?,"Based on the retrieved complaint excerpts, the issues that come up most with Savings Accounts are:\n\n* Lack of communication and warning about account issues, such as closures or changes to inter...","#9788283: . the lack of warning and communications for check deposit issues and account closures. 3. the overl...; #8475433: . my entire savings have been drained, without ample warning or follow ...",,
4,Do complaints mention long resolution times?,"Yes, complaints mention long resolution times. About 3 out of the 5 excerpts provided mention wait times or resolution times that are perceived as long. Excerpt 1 mentions no definitive time for r...","#3913191: . i called usaa i was routed to a resolution specialist who patiently listened then told me she woul...; #6876283: . every time i call, the only response i ever get is to call back in 10...",,
5,Are there complaints about hidden fees across any product?,"Yes, there are complaints about hidden fees across different products. Around 4 out of 5 excerpts mention issues with unexpected or undisclosed fees. Customers express frustration with being charg...","#7420070: . the lack of transparency and justification for these fees is deeply concerning. my attempts to com...; #2582351: . by charging excessive and undisclosed fees, i believe that they are i...",,
6,What fraud-related patterns appear in the complaints?,"Based on the retrieved complaint excerpts, two patterns appear:\n\n1. **Customers feel they've been unfairly held liable for fraudulent charges**: About 4 out of 5 excerpts (Excerpts 1, 2, 3, and ...","#3204294: . i have been a long time customer of and i have had an excellent payment history and i believe that...; #7939031: dear consumer financial protection bureau, i am writing to file a forma...",,
7,How do customers describe problems with customer service?,"Customers describe problems with customer service as being unhelpful, uncaring, and unresponsive to their concerns. About 4 out of 5 excerpts mention a lack of concern for customer satisfaction, w...",#10280939: . i think a lot of their practices with customer service should be investigated thank you for your t...; #11994279: . they do not show any concern for customer satisfaction and no drive...,,
8,Are there recurring complaints about a specific company?,"Yes, there are recurring complaints about a specific company, although the company name is not explicitly mentioned in all excerpts. Around 4-5 excerpts 

### Export the evaluation table to Markdown for the report

In [13]:
markdown_table = eval_df.to_markdown(index=False)

with open("../evaluation/eval_questions.md", "w") as f:
    f.write("# RAG Pipeline — Qualitative Evaluation\n\n")
    f.write(markdown_table)

print("Saved to ../evaluation/eval_questions.md — open it, read each answer, and fill in Quality Score + Comments.")

Saved to ../evaluation/eval_questions.md — open it, read each answer, and fill in Quality Score + Comments.


## 7. Failure case check

A grounded RAG system should say "I don't have enough information" rather than
hallucinate when the vector store has nothing relevant. Test that here with an
out-of-scope question.


In [14]:
off_topic_result = pipeline.ask("What is the weather forecast for tomorrow?")
display(Markdown(f"**Answer:**\n\n{off_topic_result['answer']}"))
print(f"\nSources used: {len(off_topic_result['sources'])}")

**Answer:**

There is no information about the weather forecast for tomorrow in the provided complaint excerpts.


Sources used: 5


## Summary for the report

- **Retriever:** top-k=5 cosine similarity search over `all-MiniLM-L6-v2` embeddings
  in ChromaDB, with optional product-category filtering.
- **Prompt design:** strict analyst persona, context-only answers, explicit
  instruction to admit insufficient evidence rather than guess.
- **Generator:** Groq Llama 3.3 70B Versatile, temperature=0.2 for consistent,
  grounded output.
- **Evaluation:** 10-question qualitative table exported to
  `evaluation/eval_questions.md` — [fill in overall takeaways once scored, e.g.
  "X/10 answers scored 4+, retrieval quality was the main limiter on Y, generation
  correctly declined to answer Z out-of-scope question"].
